# FTS Indexes

## Overview
The GIN (Generalised Inverted Index) is the standard index for PostgreSQL full-text search. It stores a posting list per lexeme — for each word, the list of documents containing it. This enables O(log N) lookup for any tsquery.

**Index choice:**

| Index | Build | Query | Size | Best for |
|---|---|---|---|---|
| GIN | Slow | Fast | Larger | Read-heavy, static/slow-changing data |
| GiST | Fast | Slower | Smaller | Write-heavy tables |

**The three storage approaches (in order of preference):**
1. `GENERATED ALWAYS AS (to_tsvector(...)) STORED` — simplest, auto-maintained
2. Trigger-maintained tsvector column — needed for multi-field weighted vectors
3. Expression index on `to_tsvector(...)` — no extra column, but expression must match exactly

---

In [1]:
import sqlite3, json

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

conn.executescript("""
-- Clinical notes and patient records for FTS demos
CREATE TABLE patients (
    patient_id  TEXT PRIMARY KEY,
    full_name   TEXT NOT NULL,
    province    TEXT NOT NULL,
    dob         TEXT NOT NULL
);

CREATE TABLE clinical_notes (
    note_id     TEXT PRIMARY KEY,
    patient_id  TEXT NOT NULL REFERENCES patients(patient_id),
    note_date   TEXT NOT NULL,
    provider    TEXT NOT NULL,
    note_type   TEXT NOT NULL,  -- 'progress','discharge','consult','operative'
    content     TEXT NOT NULL
);

CREATE TABLE medications (
    med_id      TEXT PRIMARY KEY,
    patient_id  TEXT NOT NULL,
    drug_name   TEXT NOT NULL,
    indication  TEXT,
    notes       TEXT
);

CREATE TABLE research_articles (
    article_id  TEXT PRIMARY KEY,
    title       TEXT NOT NULL,
    abstract    TEXT NOT NULL,
    keywords    TEXT,
    published   TEXT NOT NULL
);

-- FTS5 virtual tables (SQLite equivalent of PostgreSQL tsvector index)
CREATE VIRTUAL TABLE notes_fts USING fts5(
    note_id UNINDEXED,
    patient_id UNINDEXED,
    note_type UNINDEXED,
    content,
    tokenize='porter ascii'   -- porter stemmer (closest to pg's english dictionary)
);

CREATE VIRTUAL TABLE articles_fts USING fts5(
    article_id UNINDEXED,
    title,
    abstract,
    keywords,
    tokenize='porter ascii'
);
""")

patients = [
    ('P001','Alice Ngata',      'NB','1980-03-15'),
    ('P002','Bob Chen',         'ON','1972-07-22'),
    ('P003','Fatima Rashid',    'BC','1986-11-05'),
    ('P004','James MacLeod',    'NS','1963-01-30'),
    ('P005','Mei-Ling Tran',    'QC','1966-08-19'),
]
conn.executemany("INSERT INTO patients VALUES (?,?,?,?)", patients)

notes = [
    ('N001','P001','2023-10-01','Dr. Lee','progress',
     'Patient presents with poorly controlled hypertension. Blood pressure 148/92. '
     'Currently on Lisinopril 10mg once daily. Discussed medication adherence and sodium restriction. '
     'Patient reports occasional dizziness and dry cough, a known side effect of ACE inhibitors. '
     'Recommended dietary changes and increased physical activity. Follow-up in 4 weeks.'),
    ('N002','P001','2024-01-15','Dr. Lee','progress',
     'Follow-up for hypertension management. Blood pressure improved to 132/84. '
     'Patient adherent to Lisinopril. Dry cough persists. Considering switching to ARB if cough continues. '
     'HbA1c 7.2%, borderline elevated. Discussed diabetes prevention strategies. '
     'Lipid panel ordered. Continue current antihypertensive therapy.'),
    ('N003','P002','2024-01-10','Dr. Pham','discharge',
     'Patient admitted for acute exacerbation of Type 2 Diabetes. HbA1c 8.4% on admission. '
     'Adjusted Metformin to 1000mg BID and added Empagliflozin 10mg daily. '
     'Nutritional counselling provided. Patient educated on carbohydrate counting and glucose monitoring. '
     'Discharged in stable condition with follow-up appointment in 2 weeks.'),
    ('N004','P003','2023-08-20','Dr. Singh','consult',
     'Respiratory consultation for persistent asthma exacerbations. '
     'Patient reports frequent nocturnal wheezing and dyspnoea on exertion. '
     'Peak flow measurements show significant variability. Spirometry confirms obstructive pattern. '
     'Current inhaler technique reviewed and corrected. Prescribed Fluticasone-Salmeterol combination inhaler. '
     'Referral for pulmonary rehabilitation. Advised to avoid known allergens including dust mites and pet dander.'),
    ('N005','P004','2023-09-01','Dr. Lee','progress',
     'Routine diabetic review. HbA1c 7.8%, improved from last visit. '
     'eGFR 72 mL/min, stable kidney function. Patient reports no symptoms of hypoglycaemia. '
     'Foot examination normal. Retinal screening up to date. '
     'Continue current diabetes management. Annual flu vaccination administered. Blood pressure well controlled.'),
    ('N006','P005','2024-02-01','Dr. Pham','progress',
     'Complex case: Type 2 Diabetes with Hypertension and CKD Stage 3. '
     'HbA1c 9.1%, suboptimal glycaemic control. eGFR 38 mL/min, declining renal function. '
     'Metformin contraindicated due to renal impairment. Switched to Insulin glargine 20 units nocte. '
     'Potassium 5.8 mmol/L, hyperkalaemia noted. Dietary potassium restriction advised. '
     'Nephrology referral placed. Strict blood pressure control essential to slow CKD progression.'),
    ('N007','P002','2023-05-15','Dr. Pham','operative',
     'Pre-operative assessment for elective cholecystectomy. '
     'Medical history: Type 2 Diabetes, Hypertension, well controlled on Metformin and Lisinopril. '
     'Electrocardiogram normal. Chest X-ray clear. Blood glucose within acceptable range. '
     'Patient advised to withhold Metformin 48 hours prior to surgery due to contrast risk. '
     'Anaesthesia consultation arranged. Patient consented and cleared for surgery.'),
    ('N008','P001','2024-03-15','Dr. Singh','consult',
     'Cardiology consult for management of resistant hypertension. '
     'Despite optimal doses of Lisinopril and Amlodipine, blood pressure remains elevated at 158/96. '
     'Echocardiogram shows mild left ventricular hypertrophy. '
     'Added Spironolactone 25mg daily for additional blood pressure reduction. '
     'Stress test recommended to rule out ischaemic heart disease. '
     'Patient to monitor blood pressure at home twice daily and maintain log.'),
]
conn.executemany("INSERT INTO clinical_notes VALUES (?,?,?,?,?,?)", notes)

articles = [
    ('A001','Management of Type 2 Diabetes with Renal Impairment',
     'Type 2 diabetes mellitus complicated by chronic kidney disease requires careful medication selection. '
     'Metformin should be avoided when eGFR falls below 30 mL/min due to risk of lactic acidosis. '
     'SGLT2 inhibitors demonstrate renoprotective effects and cardiovascular benefits. '
     'Insulin therapy remains an option at all stages of renal impairment.',
     'diabetes,CKD,renal impairment,Metformin,SGLT2 inhibitor','2023-06-01'),
    ('A002','Hypertension Treatment Guidelines: ACE Inhibitors and ARBs',
     'Angiotensin-converting enzyme inhibitors and angiotensin receptor blockers are first-line '
     'antihypertensive agents. ACE inhibitors commonly cause dry cough due to bradykinin accumulation. '
     'ARBs are preferred in patients who cannot tolerate ACE inhibitor cough. '
     'Both drug classes provide renoprotection in diabetic nephropathy.',
     'hypertension,ACE inhibitor,ARB,Lisinopril,blood pressure','2022-11-15'),
    ('A003','Asthma Exacerbation Management in Primary Care',
     'Acute asthma exacerbations require rapid assessment of severity using peak expiratory flow and '
     'oxygen saturation. Short-acting beta-agonists remain the cornerstone of acute bronchodilation. '
     'Inhaled corticosteroids reduce airway inflammation and prevent future exacerbations. '
     'Patients with frequent exacerbations should be referred for specialist pulmonary assessment.',
     'asthma,exacerbation,bronchodilator,corticosteroid,peak flow','2023-03-20'),
    ('A004','Glycaemic Control Targets in Hospitalised Patients',
     'Inpatient hyperglycaemia is associated with increased morbidity, infection risk, and length of stay. '
     'Target blood glucose range of 6-10 mmol/L is recommended for most hospitalised patients. '
     'Insulin infusion protocols allow precise glycaemic management in critically ill patients. '
     'HbA1c measurement on admission provides information about pre-admission glycaemic control.',
     'glycaemic control,HbA1c,insulin,hospitalisation,hyperglycaemia','2024-01-08'),
    ('A005','Chronic Kidney Disease Progression and Blood Pressure Control',
     'Hypertension is both a cause and consequence of chronic kidney disease. '
     'Strict blood pressure control below 130/80 mmHg slows CKD progression significantly. '
     'Renin-angiotensin-aldosterone system blockade with ACE inhibitors or ARBs is recommended '
     'for patients with proteinuria. Regular monitoring of eGFR and potassium is essential.',
     'CKD,hypertension,blood pressure,ACE inhibitor,eGFR,proteinuria','2023-09-12'),
]
conn.executemany("INSERT INTO research_articles VALUES (?,?,?,?,?)", articles)

meds = [
    ('M001','P001','Lisinopril',   'Hypertension','10mg once daily'),
    ('M002','P001','Amlodipine',   'Hypertension','5mg once daily'),
    ('M003','P002','Metformin',    'Type 2 Diabetes','500mg BID'),
    ('M004','P002','Lisinopril',   'Hypertension','5mg once daily'),
    ('M005','P003','Salbutamol',   'Asthma','PRN inhaler'),
    ('M006','P005','Insulin glargine','Type 2 Diabetes','20 units nocte'),
    ('M007','P004','Metformin',    'Type 2 Diabetes','500mg daily'),
]
conn.executemany("INSERT INTO medications VALUES (?,?,?,?,?)", meds)

# Populate FTS5 indexes
conn.execute("INSERT INTO notes_fts SELECT note_id,patient_id,note_type,content FROM clinical_notes")
conn.execute("""
    INSERT INTO articles_fts
    SELECT article_id, title, abstract, COALESCE(keywords,'') FROM research_articles
""")
conn.commit()

print("FTS healthcare dataset loaded:")
for tbl in ['patients','clinical_notes','medications','research_articles']:
    n = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl:<22s}: {n} rows")
print("  notes_fts (FTS5)      : indexed")
print("  articles_fts (FTS5)   : indexed")


FTS healthcare dataset loaded:
  patients              : 5 rows
  clinical_notes        : 8 rows
  medications           : 7 rows
  research_articles     : 5 rows
  notes_fts (FTS5)      : indexed
  articles_fts (FTS5)   : indexed


---
## GIN vs GiST: index types

In [2]:
print("=== GIN vs GiST indexes for full-text search ===")
print()

comparison = [
    ("Build speed",    "Slower (sorts and deduplicates all postings)", "Faster"),
    ("Query speed",    "Faster (direct posting list lookup)",          "Slower (traverse tree)"),
    ("Index size",     "Larger (stores all lexeme postings)",          "Smaller (lossy, with false positives)"),
    ("Update speed",   "Slower (must update posting lists)",           "Faster (just update tree)"),
    ("False positives","None (exact)",                                 "Possible (recheck needed)"),
    ("Best for",       "Read-heavy search (static/slow-changing data)","Write-heavy tables"),
    ("Concurrency",    "Supports fast-update (batch inserts)",         "Standard"),
]
print(f"  {'Aspect':<16s}  {'GIN':<48s}  GiST")
print("  " + "-"*80)
for aspect, gin, gist in comparison:
    print(f"  {aspect:<16s}  {gin:<48s}  {gist}")

print()
print("Creating GIN and GiST indexes:")
print("""
  -- GIN (recommended for most FTS workloads):
  CREATE INDEX idx_notes_tsv_gin ON clinical_notes USING GIN (tsv);

  -- GiST (better for write-heavy tables):
  CREATE INDEX idx_notes_tsv_gist ON clinical_notes USING GiST (tsv);

  -- GIN with fast-update (batches inserts for better write performance):
  CREATE INDEX idx_notes_tsv_gin ON clinical_notes
      USING GIN (tsv) WITH (fastupdate = on);
  -- Caveat: fast-update queue must be flushed before reads see new rows
  -- Flush manually: SELECT gin_clean_pending_list('idx_notes_tsv_gin');

  -- For to_tsvector() called inline (no stored column):
  CREATE INDEX idx_notes_fts ON clinical_notes
      USING GIN (to_tsvector('english', content));
  -- Query MUST use exactly: to_tsvector('english', content) @@ ...
  -- Any variation (different config, different expression) won't use this index
""")


=== GIN vs GiST indexes for full-text search ===

  Aspect            GIN                                               GiST
  --------------------------------------------------------------------------------
  Build speed       Slower (sorts and deduplicates all postings)      Faster
  Query speed       Faster (direct posting list lookup)               Slower (traverse tree)
  Index size        Larger (stores all lexeme postings)               Smaller (lossy, with false positives)
  Update speed      Slower (must update posting lists)                Faster (just update tree)
  False positives   None (exact)                                      Possible (recheck needed)
  Best for          Read-heavy search (static/slow-changing data)     Write-heavy tables
  Concurrency       Supports fast-update (batch inserts)              Standard

Creating GIN and GiST indexes:

  -- GIN (recommended for most FTS workloads):
  CREATE INDEX idx_notes_tsv_gin ON clinical_notes USING GIN (tsv);

  -- 

---
## Stored tsvector vs inline: performance comparison

In [3]:
print("=== FTS performance: stored tsvector vs inline ===")
print()

print("Three approaches to indexing FTS (trade-offs):")
approaches = [
    ("Inline tsvector (no stored column)",
     """SELECT * FROM clinical_notes
WHERE to_tsvector('english', content) @@ query""",
     ["No extra storage", "No maintenance"],
     ["Function called on every row during index creation",
      "Expression index: query must match EXACTLY",
      "Cannot combine multiple fields easily"]),
    ("Stored tsvector column + GIN index",
     """ALTER TABLE clinical_notes ADD COLUMN tsv tsvector
    GENERATED ALWAYS AS (to_tsvector('english', content)) STORED;
CREATE INDEX ON clinical_notes USING GIN (tsv);
-- Query:
WHERE tsv @@ query""",
     ["Fastest queries (pre-computed)",
      "Clean query syntax",
      "Easy to combine fields with setweight()"],
     ["Extra storage (tsvector is typically 0.5-2x text size)",
      "Slightly slower INSERTs/UPDATEs (tsvector recomputed on write)"]),
    ("Separate FTS table",
     """CREATE TABLE notes_search (
  note_id TEXT REFERENCES clinical_notes,
  tsv     tsvector
);
-- Populated via trigger or ETL""",
     ["No changes to source table",
      "Can have custom FTS logic separate from source"],
     ["Extra complexity", "Must keep in sync with source table",
      "Two-step join for full results"]),
]
for name, code, pros, cons in approaches:
    print(f"  {name}:")
    for line in code.strip().split('\n'):
        print(f"    {line}")
    print(f"  Pros: {'; '.join(pros)}")
    print(f"  Cons: {'; '.join(cons)}")
    print()

print("EXPLAIN ANALYZE for FTS (what good looks like):")
print("""
  EXPLAIN ANALYZE
  SELECT note_id FROM clinical_notes
  WHERE tsv @@ to_tsquery('english', 'diabetes');

  -- Good plan:
  Bitmap Index Scan on idx_notes_tsv_gin  (cost=0.00..8.02 rows=1 width=0)
    Index Cond: (tsv @@ to_tsquery('english'::regconfig, 'diabetes'))
  Bitmap Heap Scan on clinical_notes  (cost=8.27..12.29 rows=1 width=32)

  -- Bad plan (no index used):
  Seq Scan on clinical_notes  (cost=0.00..18.50 rows=1 width=32)
    Filter: (to_tsvector('english', content) @@ ...)
  -- → create GIN index on stored tsvector column
""")


=== FTS performance: stored tsvector vs inline ===

Three approaches to indexing FTS (trade-offs):
  Inline tsvector (no stored column):
    SELECT * FROM clinical_notes
    WHERE to_tsvector('english', content) @@ query
  Pros: No extra storage; No maintenance
  Cons: Function called on every row during index creation; Expression index: query must match EXACTLY; Cannot combine multiple fields easily

  Stored tsvector column + GIN index:
    ALTER TABLE clinical_notes ADD COLUMN tsv tsvector
        GENERATED ALWAYS AS (to_tsvector('english', content)) STORED;
    CREATE INDEX ON clinical_notes USING GIN (tsv);
    -- Query:
    WHERE tsv @@ query
  Pros: Fastest queries (pre-computed); Clean query syntax; Easy to combine fields with setweight()
  Cons: Extra storage (tsvector is typically 0.5-2x text size); Slightly slower INSERTs/UPDATEs (tsvector recomputed on write)

  Separate FTS table:
    CREATE TABLE notes_search (
      note_id TEXT REFERENCES clinical_notes,
      tsv     t

---
## Maintaining FTS indexes: triggers and batch updates

In [4]:
print("=== Maintaining FTS indexes: triggers and updates ===")
print()

print("Keeping stored tsvector columns up to date:")
print()
print("Option 1: GENERATED ALWAYS AS (PostgreSQL 12+, simplest):")
print("""
  ALTER TABLE clinical_notes
      ADD COLUMN tsv tsvector
      GENERATED ALWAYS AS (
          to_tsvector('english', content)
      ) STORED;
  -- Auto-updated on every INSERT and UPDATE. No trigger needed.
""")

print("Option 2: Trigger (needed for multi-column tsvector):")
print("""
  CREATE OR REPLACE FUNCTION update_notes_tsv() RETURNS trigger AS $$
  BEGIN
      NEW.tsv :=
          setweight(to_tsvector('english', COALESCE(NEW.note_type,'')), 'B') ||
          setweight(to_tsvector('english', COALESCE(NEW.content,'')),   'C');
      RETURN NEW;
  END;
  $$ LANGUAGE plpgsql;

  CREATE TRIGGER trig_notes_tsv
      BEFORE INSERT OR UPDATE ON clinical_notes
      FOR EACH ROW EXECUTE FUNCTION update_notes_tsv();
""")

print("Option 3: Batch update (for large table initial population):")
print("""
  -- Initial population in batches (avoid long locks):
  DO $$
  DECLARE batch_size INT := 10000;
          offset_val INT := 0;
          rows_done  INT;
  BEGIN
      LOOP
          UPDATE clinical_notes
          SET    tsv = to_tsvector('english', content)
          WHERE  note_id IN (
              SELECT note_id FROM clinical_notes
              WHERE  tsv IS NULL
              LIMIT  batch_size
          );
          GET DIAGNOSTICS rows_done = ROW_COUNT;
          EXIT WHEN rows_done = 0;
          PERFORM pg_sleep(0.1);  -- brief pause between batches
      END LOOP;
  END $$;
""")

print("Monitoring GIN fast-update queue:")
print("""
  -- Check pending GIN items (large queue → slow queries until flushed):
  SELECT relname, n_tup_ins, n_tup_upd, n_tup_del
  FROM   pg_stat_user_tables
  WHERE  relname = 'clinical_notes';

  -- Force flush of fast-update queue:
  SELECT gin_clean_pending_list('idx_notes_tsv_gin'::regclass);
""")


=== Maintaining FTS indexes: triggers and updates ===

Keeping stored tsvector columns up to date:

Option 1: GENERATED ALWAYS AS (PostgreSQL 12+, simplest):

  ALTER TABLE clinical_notes
      ADD COLUMN tsv tsvector
      GENERATED ALWAYS AS (
          to_tsvector('english', content)
      ) STORED;
  -- Auto-updated on every INSERT and UPDATE. No trigger needed.

Option 2: Trigger (needed for multi-column tsvector):

  CREATE OR REPLACE FUNCTION update_notes_tsv() RETURNS trigger AS $$
  BEGIN
      NEW.tsv :=
          setweight(to_tsvector('english', COALESCE(NEW.note_type,'')), 'B') ||
          setweight(to_tsvector('english', COALESCE(NEW.content,'')),   'C');
      RETURN NEW;
  END;
  $$ LANGUAGE plpgsql;

  CREATE TRIGGER trig_notes_tsv
      BEFORE INSERT OR UPDATE ON clinical_notes
      FOR EACH ROW EXECUTE FUNCTION update_notes_tsv();

Option 3: Batch update (for large table initial population):

  -- Initial population in batches (avoid long locks):
  DO $$
  DECLARE b

---
## Common Pitfalls

**1. Creating a GiST index when GIN is almost always better for FTS**
GiST is lossy (false positives require recheck) and slower for reads. The only case to prefer GiST over GIN for FTS is a table with extremely high write volume where GIN's posting list maintenance is a bottleneck. For all typical analytical and clinical workloads, use GIN.

**2. GIN fastupdate queue building up and causing slow queries**
With `fastupdate = on` (default), new rows go to a pending list rather than being merged into the GIN structure immediately. A large pending list means queries must scan it linearly, negating the index benefit. Monitor with `pg_stat_user_indexes` and run `gin_clean_pending_list()` after bulk loads.

**3. Index not used because stored tsvector column is stale**
If the tsvector column is populated by a trigger and the trigger was disabled during a bulk load, new rows have `tsv = NULL` and won't match any search. After bulk loads with triggers disabled, always run `UPDATE clinical_notes SET tsv = to_tsvector('english', content) WHERE tsv IS NULL`.

**4. Creating GIN index on a table with frequent UPDATEs to indexed column**
Every UPDATE to `content` causes the GIN index to delete old postings and insert new ones — expensive for a write-heavy table. For tables where notes are frequently edited, consider using GiST with fastupdate or a separate search index table refreshed on a schedule.

**5. VACUUM not running after large deletes from GIN-indexed table**
Deleted rows leave dead postings in the GIN structure. Without regular VACUUM, the index grows with dead entries, slowing queries. Ensure autovacuum is enabled and appropriately tuned for tables with heavy FTS usage.

**6. Using pg_trgm GIN index instead of tsvector GIN for word-level search**
`CREATE INDEX ON t USING GIN (content gin_trgm_ops)` indexes trigrams, not words. It supports ILIKE and SIMILAR TO queries but does not support `@@` tsquery matching, stemming, or ranking. Use tsvector GIN for FTS; use trigram GIN for fuzzy substring matching. They solve different problems.


---
*sql_methods_library - Samantha McGarrigle*